# Signal or Noise in Multi-Agent LLM-based Stock Recommendations?

**Authors:** George Fatouros, Kostas Metaxas
**Published:** 2026-04-19
**ArXiv:** [https://arxiv.org/abs/2604.17327](https://arxiv.org/abs/2604.17327)

## Abstract
We present the first portfolio-level validation of MarketSenseAI, a deployed multi-agent LLM equity system. All signals are generated live at each observation date, eliminating look-ahead bias. The system routes four specialist agents (News, Fundamentals, Dynamics, and Macro) through a synthesis agent that issues a monthly equity thesis and recommendation for each stock in its coverage universe, and we ask two questions: do its buy recommendations add value over both passive benchmarks and random selection, and what does the internal agent structure reveal about the source of the edge? On the S&P 500 cohort (19 months) the strong-buy equal-weight portfolio earns +2.18%/month against a passive equal-weight benchmark of +1.15% (approximating RSP), a +25.2% compound excess, and ranks at the 99.7th percentile of 10,000 Monte Carlo portfolios (p=0.003). The S&P 100 cohort (35 months) delivers a +30.5% compound excess over EQWL with consistent direction but formal significance not reached, limited by the small average selection of ~10 stocks per month. Non-negative least-squares projection of thesis embeddings onto agent embeddings reveals an adaptive-integration mechanism. Agent contributions rotate with market regime (Fundamentals leads on S&P 500, Macro on S&P 100, Dynamics acts as an episodic momentum signal) and this agent rotation moves in lockstep with both the sector composition of strong-buy selections and identifiable macro-calendar events, three independent views of the same underlying adaptation. The recommendation's cross-sectional Information Coefficient is statistically significant on S&P 500 (ICIR=+0.489, p=0.024). These results suggest that multi-agent LLM equity systems can identify sources of alpha beyond what classical factor models capture, and that the buy signal functions as an effective universe-filter that can sit upstream of any portfolio-construction process.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In this phase, we define the configuration parameters for our trading strategy. This includes the universe of stocks to consider, any relevant parameters for the strategy, and a hypothesis comment block to outline our expectations and objectives.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']  # Example universe, should be expanded
HYPOTHESIS = "We hypothesize that the multi-agent LLM-based stock recommendations will outperform a passive equal-weight benchmark."

## Phase 2 — Data Download & Feature Computation

Here, we download the necessary market data using `yfinance`, compute any required factors or features, and perform cross-sectional normalization.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start='2020-01-01', end='2023-01-01', group_by='ticker')

# Example feature computation: Simple Moving Average (SMA)
def compute_sma(data, window=20):
    sma = data['Close'].rolling(window=window).mean()
    return sma

# Normalize features cross-sectionally
def normalize_features(features):
    return (features - features.mean()) / features.std()

# Apply feature computation and normalization
sma_features = {ticker: compute_sma(data[ticker]) for ticker in UNIVERSE}
normalized_sma_features = {ticker: normalize_features(sma) for ticker, sma in sma_features.items()}

## Phase 3 — Signal Generation, Position Sizing, & Portfolio Construction

In this phase, we generate trading signals based on our features, determine position sizes, and construct the portfolio.

In [ ]:
# Signal generation: Buy if SMA is above a threshold, otherwise sell
def generate_signals(features, threshold=0):
    signals = (features > threshold).astype(int)
    return signals

# Position sizing: Equal weight for simplicity
def position_sizing(signals):
    return signals / signals.sum()

# Portfolio construction
signals = {ticker: generate_signals(features) for ticker, features in normalized_sma_features.items()}
positions = {ticker: position_sizing(signal) for ticker, signal in signals.items()}

## Phase 4 — Vectorized Backtest

We perform a vectorized backtest of our strategy, ensuring no look-ahead bias by shifting signals forward by 1 period.

In [ ]:
# Vectorized backtest
def backtest(data, positions):
    prices = data['Close']
    returns = prices.pct_change().shift(-1)
    portfolio_returns = returns.multiply(positions).sum(axis=1)
    cumulative_returns = (1 + portfolio_returns).cumprod()
    return cumulative_returns

# Shift signals forward by 1 period to avoid look-ahead bias
shifted_positions = {ticker: pos.shift(1) for ticker, pos in positions.items()}

# Perform backtest
portfolio_returns = backtest(data, shifted_positions)

## Phase 5 — Performance Metrics

We calculate various performance metrics such as Sharpe ratio, Sortino ratio, Calmar ratio, maximum drawdown, and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
def calculate_metrics(returns):
    excess_returns = returns - returns.mean()
    sharpe = np.mean(excess_returns) / np.std(excess_returns)
    sortino = np.mean(excess_returns) / np.std(excess_returns[excess_returns < 0])
    calmar = np.mean(returns) / (returns.min() - returns.mean())
    max_drawdown = (returns.cummax() - returns).max()
    return sharpe, sortino, calmar, max_drawdown

# Calculate metrics
sharpe, sortino, calmar, max_drawdown = calculate_metrics(portfolio_returns)

# Plot equity curve
plt.plot(portfolio_returns)
plt.title('Equity Curve')
plt.show()

print(f'Sharpe Ratio: {sharpe:.2f}')
print(f'Sortino Ratio: {sortino:.2f}')
print(f'Calmar Ratio: {calmar:.2f}')
print(f'Max Drawdown: {max_drawdown:.2f}')

## Phase 6 — Monitoring Stub

We create a function that prints the daily P&L and current positions given live data.

In [ ]:
# Monitoring stub
def monitor_portfolio(data, positions):
    prices = data['Close']
    returns = prices.pct_change()
    portfolio_returns = returns.multiply(positions).sum(axis=1)
    daily_pnl = portfolio_returns.iloc[-1]
    current_positions = positions.iloc[-1]
    print(f'Daily P&L: {daily_pnl:.2f}')
    print('Current Positions:')
    print(current_positions)

# Example usage
monitor_portfolio(data, pd.DataFrame.from_dict(shifted_positions, orient='index').T)